# 10 — Data audit
Run before training. This notebook inspects local files without filling missing prices or creating benchmark outputs. Record the dataset page and license manually after checking the source.

In [ ]:
from pathlib import Path
import hashlib, os, sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'kaggle_global_indian_markets' / 'global-india-markets-macro'
DATA_PATH = Path(os.getenv('FIN_DATA_PATH', str(DATA_DIR / 'daily_market_data.csv'))).resolve()
MACRO_PATH = Path(os.getenv('FIN_MACRO_PATH', str(DATA_DIR / 'monthly_macro_data.csv'))).resolve()
assert DATA_PATH.is_relative_to(ROOT) and MACRO_PATH.is_relative_to(ROOT), 'Keep datasets inside this repository'
print('Python:', sys.version.split()[0], 'Daily:', DATA_PATH, 'Macro:', MACRO_PATH)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Place the audited Kaggle CSV at {DATA_PATH} or set FIN_DATA_PATH')

## File inventory and provenance
Inventory is local evidence only. The Kaggle page must be checked separately for license and description; do not infer the license from filenames.

In [ ]:
files = sorted(p for p in DATA_PATH.parent.iterdir() if p.is_file())
inventory = pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in files])
display(inventory)
print('Source page: https://www.kaggle.com/datasets/muzaddidafaque/global-and-indian-financial-markets-dataset-2025')
print('Kaggle API license identifier: CC BY-NC-SA 4.0; recheck the live dataset page before publication')
sources_file = DATA_PATH.parent / 'codes_and_sources.md'
if sources_file.exists(): print(sources_file.read_text(encoding='utf-8')[:4000])

## Raw schema and quality
This preserves raw missingness and duplicate evidence, which the adapter may subsequently clean.

In [ ]:
raw = pd.read_csv(DATA_PATH)
date_candidates = [c for c in raw if c.lower() == 'date']
date_col = date_candidates[0] if date_candidates else raw.columns[0]
dates = pd.to_datetime(raw[date_col], errors='coerce')
schema = pd.DataFrame({'column': raw.columns, 'dtype': raw.dtypes.astype(str).values, 'missing': raw.isna().sum().values, 'missing_pct': (100 * raw.isna().mean()).round(2).values})
display(schema)
print({'rows': len(raw), 'columns': len(raw.columns), 'date_field': date_col, 'unparsed_dates': int(dates.isna().sum()), 'duplicate_dates': int(dates.duplicated().sum()), 'duplicate_rows': int(raw.duplicated().sum()), 'first_date': str(dates.min()), 'last_date': str(dates.max())})
display(raw.head(3))

In [ ]:
if MACRO_PATH.is_file():
    macro_raw = pd.read_csv(MACRO_PATH)
    macro_dates = pd.to_datetime(macro_raw['date'], errors='coerce')
    display(pd.DataFrame({'column': macro_raw.columns, 'dtype': macro_raw.dtypes.astype(str).values, 'missing': macro_raw.isna().sum().values}))
    print({'macro_rows': len(macro_raw), 'macro_first': str(macro_dates.min()), 'macro_last': str(macro_dates.max()), 'macro_duplicate_dates': int(macro_dates.duplicated().sum())})
    print('Monthly macro release timing is unverified; exclude it from daily model inputs until availability lags are established.')

## Frequency and asset selection
Review units, price versus yield fields, non-trading days, and macro release timestamps before choosing tradable assets. A yield level is not automatically a bond total-return series.

In [ ]:
ordered = dates.dropna().sort_values()
display(ordered.diff().value_counts().head(10).rename_axis('gap').to_frame('count'))
display(pd.DataFrame({'candidate_field': [c for c in raw if c != date_col], 'example_values': [raw[c].dropna().head(3).tolist() for c in raw if c != date_col]}))

## Audit decision
Before `20_benchmark.ipynb`, document the chosen asset columns and excluded fields, verified license, coverage, and any cleaning rule. Do not substitute a different source without recording it.

## Durable dataset manifest
The runner records the audit under outputs/<run_id>/metadata/data_manifest.json. It includes raw daily and monthly schemas, hashes, missingness, duplicate counts, date coverage, asset candidates, source mapping and the Kaggle API license identifier.

In [ ]:
import json
from datetime import datetime, timezone
if not MACRO_PATH.is_file(): raise FileNotFoundError(f'Missing required monthly CSV: {MACRO_PATH}')
if not sources_file.is_file(): raise FileNotFoundError(f'Missing source mapping: {sources_file}')
def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
def describe_csv(path):
    frame = pd.read_csv(path)
    field = next((c for c in frame if c.lower() == 'date'), None)
    if field is None: raise ValueError(f'No date field in {path}')
    stamp = pd.to_datetime(frame[field], errors='coerce')
    gaps = stamp.dropna().sort_values().diff().dropna()
    median_gap = gaps.median()
    frequency = 'daily' if median_gap <= pd.Timedelta(days=3) else 'monthly' if median_gap >= pd.Timedelta(days=20) else 'irregular'
    return {'path':str(path.relative_to(ROOT)), 'bytes':int(path.stat().st_size),
            'sha256':file_digest(path), 'rows':int(len(frame)),
            'columns':frame.columns.tolist(), 'dtypes':{k:str(v) for k,v in frame.dtypes.items()},
            'date_field':field, 'first_date':str(stamp.min()), 'last_date':str(stamp.max()),
            'frequency_inferred':frequency, 'median_gap':str(median_gap),
            'unparsed_dates':int(stamp.isna().sum()), 'duplicate_timestamps':int(stamp.duplicated().sum()),
            'duplicate_rows':int(frame.duplicated().sum()),
            'missing_by_column':{k:int(v) for k,v in frame.isna().sum().items()}}
selected = [x.strip() for x in os.getenv('FIN_ASSETS','').split(',') if x.strip()]
daily_columns = pd.read_csv(DATA_PATH,nrows=0).columns.tolist()
canonical_assets = {'nifty50_close':'NIFTY50','sp500_close':'SP500','usd_inr_close':'USDINR','gold_close':'GOLD','brent_close':'BRENT'}
if selected and not set(selected).issubset(canonical_assets.values()): raise ValueError('FIN_ASSETS contains unknown canonical identifiers')
manifest = {'schema_version':'1.0','created_utc':datetime.now(timezone.utc).isoformat(),
            'dataset_source':'https://www.kaggle.com/datasets/muzaddidafaque/global-and-indian-financial-markets-dataset-2025',
            'license_identifier':'CC BY-NC-SA 4.0 (Kaggle API metadata)',
            'source_mapping':{'path':str(sources_file.relative_to(ROOT)),'sha256':file_digest(sources_file)},
            'asset_candidates':canonical_assets,
            'selected_assets':selected,
            'files':{'daily':describe_csv(DATA_PATH),'monthly':describe_csv(MACRO_PATH)},
            'macro_availability_note':'Release lags are unverified; monthly macro values are excluded from daily model features.'}
output_root = Path(os.getenv('OUTPUT_ROOT',str(ROOT/'outputs'))).resolve()
assert output_root.is_relative_to(ROOT)
run_id = os.getenv('RUN_ID','manual-audit').strip()
if not run_id or any(x in run_id for x in ('/','\\','..')): raise ValueError('Unsafe RUN_ID')
destination = output_root/run_id/'metadata'/'data_manifest.json'
destination.parent.mkdir(parents=True,exist_ok=True)
temporary = destination.with_suffix('.json.tmp')
temporary.write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
os.replace(temporary,destination)
figure_dir=output_root/run_id/'figures'/'data'; table_dir=output_root/run_id/'tables'/'data_audit'
figure_dir.mkdir(parents=True,exist_ok=True); table_dir.mkdir(parents=True,exist_ok=True)
schema.to_csv(table_dir/'daily_schema.csv',index=False); inventory.to_csv(table_dir/'file_inventory.csv',index=False)
import numpy as np
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(12,5)); schema.set_index('column')['missing_pct'].sort_values(ascending=False).plot(kind='bar',ax=ax)
ax.set(title='Daily source missingness',ylabel='Missing values (%)'); fig.tight_layout(); fig.savefig(figure_dir/'missingness.png',dpi=200); plt.close(fig)
close_fields=[c for c in raw if c.endswith('_close')]
if close_fields:
    returns=raw[close_fields].pct_change(fill_method=None).replace([np.inf,-np.inf],np.nan)
    returns.hist(bins=60,figsize=(12,8)); plt.suptitle('Close-to-close return distributions'); plt.tight_layout(); plt.savefig(figure_dir/'return_distributions.png',dpi=200); plt.close('all')
    corr=returns.corr(); fig,ax=plt.subplots(figsize=(8,7)); image=ax.imshow(corr,vmin=-1,vmax=1,cmap='coolwarm'); ax.set_xticks(range(len(corr)),corr.columns,rotation=90); ax.set_yticks(range(len(corr)),corr.index); fig.colorbar(image,ax=ax); ax.set_title('Return correlation heatmap'); fig.tight_layout(); fig.savefig(figure_dir/'return_correlations.png',dpi=200); plt.close(fig)
print('Wrote',destination)